# Offline Data Augmentation — Oral Cavity Images (Kaggle)

**Goal:** reduce the severe class imbalance of the **training split only** (OCA = 76 images) by saving augmented copies of the minority class(es) to disk, and produce figures for the presentation.

**Rules followed**
1. Only the **train** split is augmented. **Validation and test images are copied unchanged** (patient-level split preserved, no leakage).
2. Augmentation is applied **after** the split, on the **oral-cavity segmented** images.
3. Transforms are **clinically safe**: no hue change (lesion colour — red vs white — carries diagnostic meaning), black background preserved.
4. Every augmented image keeps its **parent image's patient ID and metadata** (Age, Gender, Smoking, …) in the output CSV.

**Outputs (in `/kaggle/working/`)**
- `augmented_dataset/{train,val,test}/{Healthy,Benign,OPMD,OCA}/…` — complete dataset ready for the models
- `split_assignment_augmented.csv` — original + augmented rows, with `image_path`, `is_augmented`, `parent_image`
- `class_weights.json` — class weights (computed on the augmented train split) for the weighted loss in the classification notebook
- `figures/*.png` — 300-dpi figures for PPT

## 1. Configuration — edit these paths to match your Kaggle dataset

In [ ]:
import os, glob, json, shutil, random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt

# ---- EDIT: root of your uploaded Kaggle dataset (contains the split/category folders + CSV)
INPUT_ROOT = "/kaggle/input/oral-cancer-segmented"      # <- change to your dataset name
CSV_PATH   = None     # None = auto-find the first *split*assign*.csv under INPUT_ROOT

OUT_ROOT   = "/kaggle/working"
OUT_DATA   = os.path.join(OUT_ROOT, "augmented_dataset")
FIG_DIR    = os.path.join(OUT_ROOT, "figures")

CLASSES = ["Healthy", "Benign", "OPMD", "OCA"]          # fixed order (also label index 0..3)
SPLITS  = ["train", "val", "test"]

# Number of NEW augmented copies per ORIGINAL train image.
# Default: OCA 76 -> 76 + 76*4 = 380. Other classes unchanged (weighted loss handles the rest).
AUG_PER_IMAGE = {"Healthy": 0, "Benign": 0, "OPMD": 0, "OCA": 4}

BLACK_THRESHOLD = 10     # pixel treated as background if all channels <= this value
SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Colours for figures: one fixed colour per class (same as the split-distribution figure)
CLASS_COLORS = {"Healthy": "#2a78d6", "Benign": "#eb6834", "OPMD": "#1baf7a", "OCA": "#eda100"}
BEFORE_COLOR = "#b4b2a9"   # neutral grey for "original"

os.makedirs(OUT_DATA, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

## 2. Load the CSV and map every image to its path on Kaggle
The CSV stores local Mac paths, so each image is located on Kaggle by its **file name**.

In [ ]:
REQUIRED = {"image_name", "patient_id", "category", "split"}
if CSV_PATH is None:
    all_csv = glob.glob(os.path.join(INPUT_ROOT, "**", "*.csv"), recursive=True)
    print("CSV files found:", *all_csv, sep="\n  ")
    # pick the CSV that actually has the per-image columns (not a summary table)
    cands = [p for p in all_csv
             if REQUIRED <= {c.strip() for c in pd.read_csv(p, nrows=0).columns}]
    cands.sort(key=lambda p: "assign" not in os.path.basename(p).lower())   # prefer *assignment*.csv
    assert cands, f"No CSV with columns {REQUIRED} found - set CSV_PATH manually"
    CSV_PATH = cands[0]
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df["split"] = df["split"].str.strip().str.lower()
df["category"] = df["category"].str.strip()
print(df.shape); display(df.head())

# index all image files under INPUT_ROOT by (split, category, file name) and by file name alone
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
by_name, by_key = {}, {}
for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True):
    if p.lower().endswith(IMG_EXT):
        name = os.path.basename(p)
        parts = [s.lower() for s in p.split(os.sep)]
        by_name.setdefault(name, []).append(p)
        for sp in SPLITS:
            for c in CLASSES:
                if sp in parts and c.lower() in parts:
                    by_key[(sp, c, name)] = p

def locate(row):
    # file name from dest_path (fallback: source_path, then image_name + any extension)
    for col in ["dest_path", "source_path"]:
        if col in row and isinstance(row[col], str):
            name = os.path.basename(row[col].strip())
            if (row["split"], row["category"], name) in by_key:
                return by_key[(row["split"], row["category"], name)]
            if name in by_name and len(by_name[name]) == 1:
                return by_name[name][0]
    for ext in IMG_EXT:
        name = f"{row['image_name']}{ext}"
        if (row["split"], row["category"], name) in by_key:
            return by_key[(row["split"], row["category"], name)]
    return None

df["input_path"] = df.apply(locate, axis=1)
missing = df["input_path"].isna().sum()
print(f"Images located: {len(df) - missing} / {len(df)}")
if missing:
    display(df[df["input_path"].isna()].head())
    raise FileNotFoundError("Some images were not found - check INPUT_ROOT / folder names")

## 3. Class distribution BEFORE augmentation

In [ ]:
before = df.groupby(["split", "category"]).size().unstack(fill_value=0).reindex(index=SPLITS, columns=CLASSES)
display(before)
tr = before.loc["train"]
print("Train imbalance ratio (largest / smallest class): %.1f : 1" % (tr.max() / tr.min()))
print((tr / tr.sum() * 100).round(1).astype(str) + " %")

## 4. Augmentation types used

| # | Augmentation | Parameter | Probability | Why it is used / why it is safe |
|---|---|---|---|---|
| 1 | Horizontal flip | left–right mirror | 0.5 | Lesions occur on both sides of the oral cavity; anatomy is roughly symmetric |
| 2 | Rotation | uniform −15° to +15°, black fill | 1.0 | Phone photos are taken at slightly different angles |
| 3 | Random zoom (crop + resize) | keeps 85–100 % of the frame | 1.0 | Different camera distances; ≥ 85 % so the lesion is not cropped out |
| 4 | Brightness | factor 0.8–1.2 | 1.0 | Different lighting (natural light / dental-chair light) |
| 5 | Contrast | factor 0.8–1.2 | 1.0 | Different camera sensors and exposure |
| 6 | Saturation | factor 0.85–1.15 | 1.0 | Colour intensity varies across phones — **hue is not changed** |
| 7 | White-balance shift | R and B channels × 0.95–1.05 | 1.0 | Simulates colour temperature of different phone cameras |
| 8 | Gaussian blur | radius 0.3–1.0 px | 0.2 | Slight defocus / hand movement |
| 9 | Gaussian noise | σ = 4 (0–255 scale) | 0.2 | Sensor noise in low light |
| – | Background re-masking | pixels outside the oral cavity set back to black | always | Keeps augmented images consistent with the segmented originals |

**Deliberately NOT used:** hue shift or colour inversion (would alter red/white lesion appearance and therefore the diagnosis), vertical flip (unnatural orientation), large cut-out/erasing (may delete the lesion), elastic distortion, and mixing images of different classes (mixup/cutmix).

In [ ]:
AUG_TABLE = pd.DataFrame([
    ["Horizontal flip", "left-right mirror", "0.5"],
    ["Rotation", "-15 to +15 deg, black fill", "1.0"],
    ["Random zoom", "keeps 85-100 % of frame", "1.0"],
    ["Brightness", "factor 0.8-1.2", "1.0"],
    ["Contrast", "factor 0.8-1.2", "1.0"],
    ["Saturation", "factor 0.85-1.15 (no hue change)", "1.0"],
    ["White-balance shift", "R, B channels x 0.95-1.05", "1.0"],
    ["Gaussian blur", "radius 0.3-1.0 px", "0.2"],
    ["Gaussian noise", "sigma = 4", "0.2"],
    ["Background re-masking", "non-oral pixels kept black", "always"],
], columns=["Augmentation", "Parameter range", "Probability"])
AUG_TABLE.to_csv(os.path.join(OUT_ROOT, "augmentation_types.csv"), index=False)

# table as an image for PPT
fig, ax = plt.subplots(figsize=(9, 4.2)); ax.axis("off")
tbl = ax.table(cellText=AUG_TABLE.values, colLabels=AUG_TABLE.columns, loc="center", cellLoc="left", colLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5); tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#d9d8d2")
    if r == 0: cell.set_text_props(weight="bold"); cell.set_facecolor("#f1f0ec")
ax.set_title("Augmentation types applied to the training split", fontsize=13, pad=10)
plt.savefig(os.path.join(FIG_DIR, "fig1_augmentation_types_table.png"), bbox_inches="tight", facecolor="white")
plt.show()

## 5. Augmentation functions

In [ ]:
def fg_mask(arr):
    return (arr > BLACK_THRESHOLD).any(axis=2)

def remask(img, mask):
    arr = np.asarray(img).astype(np.float32); arr[~mask] = 0
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

# ---- individual operations (also used for the "augmentation types" figure)
def op_hflip(img, r):      return img.transpose(Image.FLIP_LEFT_RIGHT)
def op_rotate(img, r, a=None):
    a = r.uniform(-15, 15) if a is None else a
    return img.rotate(a, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
def op_zoom(img, r, s=None):
    w, h = img.size; s = r.uniform(0.85, 1.0) if s is None else s
    cw, ch = int(w * s), int(h * s); x0 = r.integers(0, w - cw + 1); y0 = r.integers(0, h - ch + 1)
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((w, h), Image.BILINEAR)
def op_brightness(img, r, f=None): return ImageEnhance.Brightness(img).enhance(r.uniform(0.8, 1.2) if f is None else f)
def op_contrast(img, r, f=None):   return ImageEnhance.Contrast(img).enhance(r.uniform(0.8, 1.2) if f is None else f)
def op_saturation(img, r, f=None): return ImageEnhance.Color(img).enhance(r.uniform(0.85, 1.15) if f is None else f)
def op_white_balance(img, r, fr=None, fb=None):
    arr = np.asarray(img).astype(np.float32)
    arr[..., 0] *= r.uniform(0.95, 1.05) if fr is None else fr
    arr[..., 2] *= r.uniform(0.95, 1.05) if fb is None else fb
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
def op_blur(img, r, rad=None):     return img.filter(ImageFilter.GaussianBlur(r.uniform(0.3, 1.0) if rad is None else rad))
def op_noise(img, r, sigma=4.0):
    arr = np.asarray(img).astype(np.float32) + r.normal(0, sigma, (img.size[1], img.size[0], 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def augment(img, r=rng):
    # full random pipeline: geometry -> mask -> photometric -> re-mask background
    img = img.convert("RGB")
    if r.random() < 0.5: img = op_hflip(img, r)
    img = op_rotate(img, r)
    img = op_zoom(img, r)
    mask = fg_mask(np.asarray(img))          # oral-cavity region after geometry
    img = op_brightness(img, r); img = op_contrast(img, r); img = op_saturation(img, r)
    img = op_white_balance(img, r)
    if r.random() < 0.2: img = op_blur(img, r)
    if r.random() < 0.2: img = op_noise(img, r)
    return remask(img, mask)

## 6. Copy originals and generate augmented images (TRAIN only)

In [ ]:
rows = []
for _, row in df.iterrows():
    dst_dir = os.path.join(OUT_DATA, row["split"], row["category"]); os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(row["input_path"]))
    shutil.copy2(row["input_path"], dst)
    r0 = row.to_dict(); r0.update(image_path=dst, is_augmented=0, parent_image=row["image_name"])
    rows.append(r0)

    n = AUG_PER_IMAGE.get(row["category"], 0) if row["split"] == "train" else 0
    if n:
        img = Image.open(row["input_path"]).convert("RGB")
        stem, ext = os.path.splitext(os.path.basename(row["input_path"]))
        for k in range(1, n + 1):
            out = os.path.join(dst_dir, f"{stem}_aug{k}{ext}")
            augment(img).save(out, quality=95)
            r1 = row.to_dict()
            r1.update(image_name=f"{row['image_name']}_aug{k}", image_path=out,
                      is_augmented=1, parent_image=row["image_name"])
            rows.append(r1)

aug_df = pd.DataFrame(rows).drop(columns=["input_path"])
aug_df.to_csv(os.path.join(OUT_ROOT, "split_assignment_augmented.csv"), index=False)

# safety checks
assert set(aug_df.loc[aug_df.is_augmented == 1, "split"]) <= {"train"}, "augmented images outside train!"
tr_p = set(aug_df.loc[aug_df.split == "train", "patient_id"])
for sp in ["val", "test"]:
    assert not (tr_p & set(aug_df.loc[aug_df.split == sp, "patient_id"])), f"patient overlap train/{sp}"
print("Augmented images created:", int(aug_df.is_augmented.sum()))

after = aug_df.groupby(["split", "category"]).size().unstack(fill_value=0).reindex(index=SPLITS, columns=CLASSES)
display(pd.concat({"Before": before, "After": after}, axis=1))

## 7. Class weights for the weighted loss (saved for the classification notebook)
$w_c = \dfrac{N}{K \cdot n_c}$, computed on the **augmented train split**, normalised to mean 1.

In [ ]:
n_c = after.loc["train"].values.astype(float)
w = n_c.sum() / (len(CLASSES) * n_c); w = w / w.mean()
class_weights = {c: round(float(x), 4) for c, x in zip(CLASSES, w)}
json.dump({"classes": CLASSES, "train_counts": dict(zip(CLASSES, n_c.astype(int).tolist())),
           "class_weights": class_weights}, open(os.path.join(OUT_ROOT, "class_weights.json"), "w"), indent=2)
print(class_weights)

## 8. Figures for PPT

In [ ]:
# Fig 2 - train class counts before vs after (grouped bars)
x = np.arange(len(CLASSES)); bw = 0.38
b_tr, a_tr = before.loc["train"].values, after.loc["train"].values
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(x - bw/2 - 0.01, b_tr, bw, color=BEFORE_COLOR, label="Before augmentation")
ax.bar(x + bw/2 + 0.01, a_tr, bw, color=[CLASS_COLORS[c] for c in CLASSES], label="After augmentation")
off = max(a_tr.max(), b_tr.max()) * 0.012
for i in range(len(CLASSES)):
    ax.text(x[i] - bw/2 - 0.01, b_tr[i] + off, str(b_tr[i]), ha="center", va="bottom", fontsize=10, color="#3d3c38")
    ax.text(x[i] + bw/2 + 0.01, a_tr[i] + off, str(a_tr[i]), ha="center", va="bottom", fontsize=10, color="#0b0b0b", weight="bold")
ax.set_xticks(x, CLASSES); ax.set_ylabel("Number of training images")
ax.set_title("Training images per class: before vs after augmentation")
ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BEFORE_COLOR, label="Before augmentation"),
                   Patch(color="#6e6d68", label="After augmentation (class colour)")],
          frameon=False, loc="upper left")
ax.set_ylim(0, max(a_tr.max(), b_tr.max()) * 1.18)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig2_train_counts_before_after.png"), facecolor="white"); plt.show()

In [ ]:
# Fig 3 - class share of the training split (100% stacked bars) before vs after
fig, ax = plt.subplots(figsize=(9, 2.9))
for yi, (lab, cnt) in enumerate([("After", a_tr), ("Before", b_tr)]):
    left = 0; pct = cnt / cnt.sum() * 100
    for c, p in zip(CLASSES, pct):
        ax.barh(yi, p - 0.3, left=left, color=CLASS_COLORS[c], height=0.6)
        if p > 5:
            ax.text(left + p/2, yi, f"{c}\n{p:.1f}%", ha="center", va="center", fontsize=9.5, color="white", weight="bold")
        left += p
ax.set_yticks([0, 1], ["After", "Before"]); ax.set_xlim(0, 100); ax.set_xlabel("Share of training images (%)")
ax.set_title("Class share of the training split")
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(color=CLASS_COLORS[c], label=c) for c in CLASSES], ncol=4, frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, -0.35))
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig3_train_class_share_before_after.png"), facecolor="white"); plt.show()

In [ ]:
# Fig 4 - all splits, before vs after (val/test unchanged), same y-axis
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, (title, tab) in zip(axes, [("Before augmentation", before), ("After augmentation", after)]):
    xs = np.arange(len(SPLITS)); bw = 0.2
    for j, c in enumerate(CLASSES):
        vals = tab[c].values
        ax.bar(xs + (j - 1.5) * (bw + 0.01), vals, bw, color=CLASS_COLORS[c], label=c)
        for xi, v in zip(xs, vals):
            ax.text(xi + (j - 1.5) * (bw + 0.01), v + after.values.max() * 0.012, str(v), ha="center", fontsize=8.5, color="#3d3c38")
    ax.set_xticks(xs, [s.upper() for s in SPLITS]); ax.set_title(title)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
axes[0].set_ylabel("Number of images"); axes[1].legend(title="Category", frameon=False)
fig.suptitle("Images per split and class (patient-level split; only TRAIN is augmented)", y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig4_all_splits_before_after.png"), bbox_inches="tight", facecolor="white"); plt.show()

In [ ]:
# Fig 5 - each augmentation type applied separately to one training image
demo_cls = "OCA" if (df.split.eq("train") & df.category.eq("OCA")).any() else CLASSES[0]
demo = Image.open(df[(df.split == "train") & (df.category == demo_cls)].iloc[0]["input_path"]).convert("RGB")
m0 = fg_mask(np.asarray(demo)); r = np.random.default_rng(7)
def geo(f):  # geometric op -> recompute mask
    out = f(demo); return remask(out, fg_mask(np.asarray(out)))
panels = [("Original", demo),
          ("Horizontal flip", geo(lambda i: op_hflip(i, r))),
          ("Rotation (+15°)", geo(lambda i: op_rotate(i, r, 15))),
          ("Zoom (85%)", geo(lambda i: op_zoom(i, r, 0.85))),
          ("Brightness ×1.2", remask(op_brightness(demo, r, 1.2), m0)),
          ("Contrast ×1.2", remask(op_contrast(demo, r, 1.2), m0)),
          ("Saturation ×1.15", remask(op_saturation(demo, r, 1.15), m0)),
          ("White balance (warm)", remask(op_white_balance(demo, r, 1.05, 0.95), m0)),
          ("Gaussian blur (r=1.0)", remask(op_blur(demo, r, 1.0), m0)),
          ("Gaussian noise (σ=4)", remask(op_noise(demo, r, 4.0), m0))]
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, (t, im) in zip(axes.ravel(), panels):
    ax.imshow(im); ax.set_title(t, fontsize=11); ax.axis("off")
fig.suptitle(f"Augmentation types (each applied alone) — example {demo_cls} training image", fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig5_augmentation_types_examples.png"), facecolor="white"); plt.show()

In [ ]:
# Fig 6 - original vs saved augmented images (random pipeline), several OCA training images
parents = aug_df[(aug_df.is_augmented == 1)].parent_image.drop_duplicates()
parents = parents.sample(min(4, len(parents)), random_state=SEED).tolist()
n_aug = max(AUG_PER_IMAGE.values())
if parents:
    fig, axes = plt.subplots(len(parents), n_aug + 1, figsize=(3.1 * (n_aug + 1), 3.1 * len(parents)), squeeze=False)
    for i, p in enumerate(parents):
        orig = aug_df[(aug_df.image_name == p) & (aug_df.is_augmented == 0)].iloc[0]
        kids = aug_df[(aug_df.parent_image == p) & (aug_df.is_augmented == 1)]
        imgs = [("Original", orig.image_path)] + [(f"Augmented {k+1}", q) for k, q in enumerate(kids.image_path)]
        for j in range(n_aug + 1):
            ax = axes[i, j]; ax.axis("off")
            if j < len(imgs):
                ax.imshow(Image.open(imgs[j][1])); ax.set_title(imgs[j][0] if i == 0 else "", fontsize=12)
        axes[i, 0].text(-0.08, 0.5, f"{orig.category}\n{p}", transform=axes[i, 0].transAxes,
                        ha="right", va="center", fontsize=10)
    fig.suptitle("Before (original) and after (saved augmented copies)", fontsize=14)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig6_before_after_examples.png"), facecolor="white"); plt.show()

## 9. Zip outputs for download / reuse as a Kaggle dataset

In [ ]:
shutil.make_archive(os.path.join(OUT_ROOT, "figures"), "zip", FIG_DIR)
print(sorted(os.listdir(FIG_DIR)))
print("Dataset folder:", OUT_DATA)
# Tip: in the Kaggle 'Output' tab, click 'New Dataset' on augmented_dataset to use it in the model notebooks.